# Voice cloning on a free GPU (XTTS v2)

Clone your own voice from a short recording, then speak any text with it. Arabic and English supported.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

Only clone voices you own or have permission to use. The model is under the Coqui Public Model License (non-commercial);
you'll be asked to accept it when the model first loads.

## 1. Install (≈2 min)
If Colab asks you to restart the runtime after this, do it, then continue from step 2.

In [ ]:
!pip install -q coqui-tts

## 2. Upload your voice recording
15–30 seconds, quiet room, natural pace. Any format (m4a, mp3, wav...).

In [ ]:
from google.colab import files
import subprocess, os

up = files.upload()
src = next(iter(up))
subprocess.run(["ffmpeg", "-y", "-i", src, "-ac", "1", "-ar", "22050", "-t", "30",
                "-af", "silenceremove=start_periods=1:start_threshold=-45dB", "reference.wav"],
               check=True, capture_output=True)
print("Reference ready: reference.wav")

## 3. Load the model (first time downloads ~2 GB and asks you to type `y` to accept the license)

In [ ]:
import torch, re
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("No GPU detected. Runtime → Change runtime type → T4 GPU, then re-run from step 3.")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

ARABIC = re.compile(r"[\u0600-\u06FF]")
def detect_lang(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return "en"
    return "ar" if sum(bool(ARABIC.match(c)) for c in letters) / len(letters) > 0.5 else "en"

## 4. Say anything
Edit the text and run. Re-run this cell as many times as you like; the model stays loaded.

In [ ]:
from IPython.display import Audio, display
import time

TEXT = "Hello, this is my cloned voice speaking."
SPEED = 1.0          # 0.9 slower, 1.1 faster
OUT = "output.wav"

t0 = time.time()
tts.tts_to_file(text=TEXT, speaker_wav="reference.wav", language=detect_lang(TEXT),
                file_path=OUT, speed=SPEED, split_sentences=True)
print(f"Generated in {time.time() - t0:.1f}s")
display(Audio(OUT))
files.download(OUT)